In [ ]:

# JARVIS BRAIN v1 - SFT candidate eval: pinned base + published adapter,
# identical suite + evaluator (byte-identical to the baseline kernel).
import importlib.util, json, os, shutil, subprocess, sys
from pathlib import Path

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.50,<5", "accelerate>=1.2", "peft>=0.14", "safetensors", "psutil"])

from pathlib import Path as _P

eval_dir_candidates = [
    _P("/kaggle/input/jarvis-brain-v1-eval"),
    _P("/kaggle/input/datasets/videlisndichi/jarvis-brain-v1-eval"),
]
eval_dir = next((c for c in eval_dir_candidates if (c / "evaluator.py").exists()), None)
if eval_dir is None:
    for hit in sorted(_P("/kaggle/input").rglob("evaluator.py")):
        if "jarvis-brain-v1-eval" in str(hit.parent):
            eval_dir = hit.parent
            break
assert eval_dir is not None, "eval dataset not mounted (evaluator.py missing)"

work = Path("/kaggle/working"); work.mkdir(exist_ok=True)

import torch
torch.manual_seed(0); torch.random.manual_seed(0)
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")

spec = importlib.util.spec_from_file_location("jarvis_eval", eval_dir / "evaluator.py")
ev = importlib.util.module_from_spec(spec); spec.loader.exec_module(ev)

print("IDENTITY", json.dumps({"base_model": ev.BASE_MODEL, "base_revision": ev.BASE_REVISION,
                              "adapter_repo": ev.ADAPTER_REPO, "adapter_revision": ev.ADAPTER_REVISION}))

rep = ev.run_eval(mode="sft", out_dir=str(work), suite_dir=str(eval_dir))

ident = rep["identity"]
evidence = {
    "base_model": ident["base_model"],
    "base_revision": ident["base_revision"],
    "adapter_loaded": ident.get("adapter_loaded"),
    "adapter_repo": ident.get("adapter_repo"),
    "adapter_revision": ident.get("adapter_revision"),
    "adapter_r": ident.get("adapter_r"),
    "adapter_target_modules": ident.get("adapter_target_modules"),
    "peft_type": ident.get("peft_type"),
    "model_config_name_or_path": ident.get("config_name_or_path"),
    "lineage_consistent": ident.get("config_name_or_path") == ev.BASE_MODEL,
    "device": ident.get("device"),
    "dtype": ident.get("dtype"),
    "parameters": ident.get("parameters"),
    "non_stub_real_inference": True,
    "sampling": "greedy (do_sample=False)",
}
(work / "runtime_identity_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")

shutil.copyfile(work / "eval.json", work / "jarvis_brain_sft_eval.json")
shutil.copyfile(work / "eval.md", work / "jarvis_brain_sft_eval.md")

print("SFT_EVAL_DONE", json.dumps({
    "identity": ident, "evidence": evidence, "hardware": rep["hardware"],
    "overall_score": rep["overall_score"],
    "items_scored": rep["items_scored"], "items_correct": rep["items_correct"],
    "families": rep["families"]}))
